# Task 2 — Supervised Learning

## Objective
This notebook builds and evaluates supervised machine learning models to predict bike rental demand (`cnt`).

## Input
- ../data/cleaned.csv

## Output
- ../models/supervised_best.pkl
- model evaluation results

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

In [11]:
DATA_FILE = Path("../data/cleaned.csv")
MODEL_FILE = Path("../models/supervised_best.pkl")

df = pd.read_csv(DATA_FILE)
df.head()

,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,40
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,32
3,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13
4,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1


## Prediction Task

This is a regression task because the target variable `cnt` is continuous and represents the total number of bike rentals.

The goal is to predict rental demand based on time-related and environmental features.

In [12]:
df["temp_hum"] = df["temp"] * df["hum"]

df["is_peak_hour"] = df["hr"].apply(lambda x: 1 if x in [8, 17, 18] else 0)

## Feature Engineering

Two new features were created:

- `temp_hum`: interaction between temperature and humidity. This feature captures combined environmental effects that may influence bike rental demand.
- `is_peak_hour`: binary indicator for peak commuting hours (morning and evening). This helps the model capture daily demand patterns.

In [13]:
target = "cnt"

X = df.drop(columns=[target])
y = df[target]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (13903, 14)
Test size: (3476, 14)


In [15]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42)
}

In [16]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")

    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "CV Mean R2": cv_scores.mean()
    })

## Model Comparison

In [17]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="RMSE")

results_df

,Model,RMSE,MAE,R2,CV Mean R2
1,Decision Tree,57.686682,34.368959,0.894909,0.884673
0,Linear Regression,118.950647,91.101144,0.553164,0.557724


In [18]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)

joblib.dump(best_model, MODEL_FILE)

print(f"Best model ({best_model_name}) saved")

Best model (Decision Tree) saved


## Conclusion

The Decision Tree model performed better than Linear Regression based on the evaluation metrics. It achieved lower RMSE and MAE values, as well as a higher R² score, indicating that it provides more accurate predictions of bike rental demand.

In practical terms, this means that the model is able to estimate the number of bike rentals with smaller average error. This is important for real-world applications such as resource planning and demand forecasting in bike-sharing systems.

The improved performance of the Decision Tree suggests that the relationship between features and demand is not purely linear, and more flexible models are better suited for capturing these patterns.